### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="mutual_funds_india",
    dataset_year="2023",
    domain_str="finance",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/ravibarnawal/mutual-funds-india-detailed",
    download_description="""
kaggle datasets download ravibarnawal/mutual-funds-india-detailed --unzip && mkdir -p local-data-warehouse/mutual_funds_india && mv comprehensive_mutual_funds_data.csv local-data-warehouse/mutual_funds_india/
""",
    # References
    academic_reference_bibtex=r"""@misc{Barnawal2022MutualFundsIndiaDetailed,
  author = {Ravi Barnawal},
  title  = {Mutual Funds India Detailed},
  year   = {2022},
  howpublished = {\url{https://www.kaggle.com/datasets/ravibarnawal/mutual-funds-india-detailed}},
  note   = {Kaggle dataset}
}
""",
    academic_reference_bibtex_key="Barnawal2022MutualFundsIndiaDetailed",
    license="CC0: Public Domain",
    data_tags=["IID"],
    curation_comments="""
We start with the data from Kaggle.

- We aim to predict the return over 3 years and drop all samples that do not have data for 3 year return.
- We drop potentially leaking columns about returns (alpha, beta, sd, sharpe, sortino).
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="returns_3yr",
    problem_type="regression",
    objective_metric_name="rmse",
)

## Preprocessing

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "comprehensive_mutual_funds_data.csv")
print("Loaded data shape:", df.shape)

df = df.drop(columns=["alpha", "beta","sd", "sharpe", "sortino", "returns_5yr", "returns_1yr"])

df = df[df["returns_3yr"].notna()]

as_string_type = [
    "scheme_name",
    "amc_name",
    "fund_manager",
]
as_cat_type = [
    "sub_category",
    "category",
]
for c in as_string_type:
    nan_mask = df[c].isna()
    df.loc[nan_mask, c] = np.nan
    df[c] = df[c].astype("string")

df[as_cat_type] = df[as_cat_type].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()